# Notebook 19 — Pipeline-Level Evaluation: RF vs MobileBERT Zero-Shot

## Objective

Re-evaluate CSIC 2010 through the **full production pipeline** for both models,
using each model's best-performing input strategy:

| Model | Strategy | Best threshold | Source |
|---|---|---|---|
| RF (NB07) | Way 3 — query values joined | 0.82 | NB18 |
| MobileBERT zero-shot (cssupport) | Way 3 — query values joined | 0.99 | NB17 |

Both models use **identical input extraction** (`extract_query_values` — join parameter values)
so the comparison is apples-to-apples at the pipeline level.

## Answer to: is zero-shot the same as Way 3?

Yes. The cssupport MobileBERT model was trained on `Modified_SQL_Dataset.csv` which contains
bare SQL query strings — equivalent to query parameter values stripped of their HTTP context.
In NB17 it was evaluated on query values extracted from CSIC requests, which is the same
input format as RF Way 3. Both models receive the same text at inference time.

## Pipeline (identical to NB11)

```
Apache log line
  → parse_log_line()        (extract method, URL, status)
  → extract_query_values()  (decode + join parameter values)
  → score()                 (RF or MobileBERT)
  → tier                    (ATTACK / SUSPICIOUS / BENIGN / NO_QS)
```

## Dataset

CSIC 2010 converted to Apache Combined Log Format (same as NB11):
- `csic2010_attack_access.log`
- `csic2010_benign_access.log`

## 1. Imports & Setup

In [ ]:
import os, re, csv, time, urllib.parse, warnings, json
import numpy as np
import pandas as pd
import joblib
import torch
from collections import Counter
from scipy.sparse import hstack, csr_matrix
from transformers import MobileBertTokenizer, MobileBertForSequenceClassification

warnings.filterwarnings('ignore')

for d in ['results/models', 'results/figures', 'results/metrics']:
    os.makedirs(d, exist_ok=True)

# ── Paths ─────────────────────────────────────────────────────────
ATTACK_LOG   = '../logs/csic2010/csic2010_attack_access.log'
BENIGN_LOG   = '../logs/csic2010/csic2010_benign_access.log'

RF_MODEL_PATH   = 'results/models/07_rf_model.pkl'
RF_VEC_PATH     = 'results/models/07_vectorizer.pkl'
MB_MODEL_PATH   = 'results/models/15_mobilebert_zeroshot'  # cached from NB17

# ── Thresholds (best F1 from NB17 and NB18) ───────────────────────
# RF:          threshold=0.82, recall=0.827, FP/10k=33.6, F1=0.843
# MobileBERT:  threshold=0.99, recall=0.799, FP/10k=210.5, F1=0.604
RF_THRESHOLD = 0.82
MB_THRESHOLD = 0.99

BATCH_SIZE   = 2000   # RF batch size (CPU)
MB_BATCH     = 128    # MobileBERT batch size (GPU)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device       : {DEVICE}')
print(f'RF threshold : {RF_THRESHOLD}')
print(f'MB threshold : {MB_THRESHOLD}')
print(f'Attack log   : {ATTACK_LOG}')
print(f'Benign log   : {BENIGN_LOG}')
print()
for label, path in [
    ('Attack log',   ATTACK_LOG),
    ('Benign log',   BENIGN_LOG),
    ('RF model',     RF_MODEL_PATH),
    ('RF vectorizer', RF_VEC_PATH),
    ('MB model',     MB_MODEL_PATH),
]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // 1024 if exists else 0
    print(f'  {"✅" if exists else "❌"} {label:<16}: {size:>6} KB')
print('\nSetup complete.')

## 2. Core Pipeline Functions

Identical to NB11 — same log parser, same extraction, same symbol matrix.

In [ ]:
SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

# Apache Combined Log Format parser — identical to NB11 production pipeline
LOG_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ \[[^\]]+\] '
    r'"(?P<method>\S+) (?P<url>.+?) HTTP/\d\.\d" '
    r'(?P<status>\d{3}) (?P<bytes>\S+)'
    r'(?: "[^"]*" "[^"]*")?'
)

def parse_log_line(raw):
    m = LOG_PATTERN.match(raw.strip())
    if not m:
        return None
    d = m.groupdict()
    d['bytes']  = int(d['bytes']) if d['bytes'].isdigit() else 0
    d['status'] = int(d['status'])
    return d

def extract_query_values(url):
    """Extract and join decoded query parameter values — Way 3."""
    try:
        parsed = urllib.parse.urlparse(url)
        params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
        values = [
            urllib.parse.unquote(v).strip()
            for vlist in params.values()
            for v in vlist
            if urllib.parse.unquote(v).strip()
        ]
        return ' '.join(values) if values else None
    except Exception:
        return None

def build_symbol_matrix(queries):
    rows = []
    for q in queries:
        c = Counter()
        for sym in SYMBOLS:
            c[sym] = str(q).count(sym)
        rows.append([c[sym] for sym in SYMBOLS])
    return csr_matrix(np.array(rows, dtype=float))

def categorise(qv):
    """Attack type classifier — identical to NB11."""
    if not qv or str(qv) == 'nan':
        return 'Unknown'
    q = str(qv).lower()
    if 'waitfor' in q or 'sleep(' in q or 'pg_sleep' in q:
        return 'SQLi — Time-based'
    if 'union' in q and ('select' in q or 'all' in q):
        return 'SQLi — UNION'
    if 'select' in q and ('from' in q or 'where' in q):
        return 'SQLi — Statement'
    if 'drop table' in q or 'insert into' in q:
        return 'SQLi — DML'
    if 'script' in q or 'alert(' in q or 'javascript:' in q or \
       'paros' in q or 'document.location' in q or '<!--#' in q:
        return 'XSS / SSI'
    if "'" in q and ('--' in q or '#' in q or '/*' in q):
        return 'SQLi — Quote+Comment'
    if "'" in q and ('=' in q or ' or ' in q or ' and ' in q):
        return 'SQLi — Probe'
    if ';' in q and "'" in q:
        return 'SQLi — Stacked'
    if '../' in q or 'etc/passwd' in q:
        return 'Path Traversal'
    return 'Other'

SQLI_TYPES = [
    'SQLi — Time-based', 'SQLi — UNION', 'SQLi — Statement',
    'SQLi — DML', 'SQLi — Quote+Comment', 'SQLi — Probe', 'SQLi — Stacked',
]

print('Core functions defined.')

## 3. Load RF Model

In [ ]:
rf_model = joblib.load(RF_MODEL_PATH)
rf_vec   = joblib.load(RF_VEC_PATH)

print(f'RF model    : {type(rf_model).__name__}')
print(f'Vocab       : {len(rf_vec.vocabulary_):,} features')
print(f'Features    : {rf_model.n_features_in_:,} (n-gram + symbol)')
print(f'Threshold   : {RF_THRESHOLD}')

# Sanity check
test = [
    "'; DROP TABLE usuarios; SELECT * FROM datos WHERE nombre LIKE '%",
    "Jamón Ibérico",
    "1','0','0');waitfor delay '0:0:15';--",
]
ngram = rf_vec.transform(test)
sym   = build_symbol_matrix(test)
probs = rf_model.predict_proba(hstack([ngram, sym]))[:, 1]
print()
print('RF sanity check:')
for t, p in zip(test, probs):
    print(f'  {p:.4f}  {t[:70]}')

## 4. Load MobileBERT Zero-Shot Model

In [ ]:
MB_BASE_TOKENIZER = 'google/mobilebert-uncased'
MB_HF_MODEL       = 'cssupport/mobilebert-sql-injection-detect'

if os.path.exists(MB_MODEL_PATH):
    print(f'Loading MobileBERT from cache: {MB_MODEL_PATH}')
    mb_tokenizer = MobileBertTokenizer.from_pretrained(MB_BASE_TOKENIZER)
    mb_model     = MobileBertForSequenceClassification.from_pretrained(MB_MODEL_PATH)
else:
    print(f'Downloading {MB_HF_MODEL} from HuggingFace Hub...')
    mb_tokenizer = MobileBertTokenizer.from_pretrained(MB_BASE_TOKENIZER)
    mb_model     = MobileBertForSequenceClassification.from_pretrained(MB_HF_MODEL)
    os.makedirs(MB_MODEL_PATH, exist_ok=True)
    mb_model.save_pretrained(MB_MODEL_PATH)
    print(f'  Cached: {MB_MODEL_PATH}')

mb_model.to(DEVICE)
mb_model.eval()

print(f'MobileBERT loaded on {DEVICE}')
print(f'Threshold  : {MB_THRESHOLD}')

# Sanity check
print()
print('MobileBERT sanity check:')
for t in test:
    inputs = mb_tokenizer(t, truncation=True, padding=True,
                          max_length=128, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        prob = torch.softmax(mb_model(**inputs).logits, dim=1)[0, 1].item()
    print(f'  {prob:.4f}  {t[:70]}')

## 5. Pipeline Evaluation Functions

In [ ]:
# ── RF pipeline evaluator ─────────────────────────────────────────
def evaluate_rf(log_path, label, output_path, threshold):
    """
    Process log file through RF pipeline — identical to NB11.
    Uses combined feature matrix: hstack([tfidf_ngram, symbol_counts]).
    """
    counts   = {'total': 0, 'parse_errors': 0,
                'no_qs': 0, 'scored': 0,
                'attack': 0, 'benign': 0}
    buf_rows = []
    buf_qvs  = []

    def flush(writer):
        if not buf_qvs:
            return
        ngram = rf_vec.transform(buf_qvs)
        sym   = build_symbol_matrix(buf_qvs)
        probs = rf_model.predict_proba(hstack([ngram, sym]))[:, 1]
        for row, prob in zip(buf_rows, probs):
            prob = round(float(prob), 6)
            tier = 'ATTACK' if prob >= threshold else 'BENIGN'
            counts['attack' if tier == 'ATTACK' else 'benign'] += 1
            counts['scored'] += 1
            writer.writerow({'label': label, 'url': row[0], 'qv': row[1],
                             'score': prob, 'tier': tier,
                             'method': row[2], 'status': row[3]})
        buf_rows.clear()
        buf_qvs.clear()

    with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile,
            fieldnames=['label', 'url', 'qv', 'score', 'tier', 'method', 'status'])
        writer.writeheader()
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            for raw_line in f:
                parsed = parse_log_line(raw_line)
                if parsed is None:
                    counts['parse_errors'] += 1
                    continue
                qv = extract_query_values(parsed['url'])
                counts['total'] += 1
                if qv is None:
                    counts['no_qs'] += 1
                else:
                    buf_rows.append((parsed['url'], qv, parsed['method'], parsed['status']))
                    buf_qvs.append(qv)
                    if len(buf_qvs) >= BATCH_SIZE:
                        flush(writer)
        flush(writer)
    return counts


# ── MobileBERT pipeline evaluator ────────────────────────────────
def evaluate_mobilebert(log_path, label, output_path, threshold):
    """
    Process log file through MobileBERT pipeline.
    Input: extracted query values (Way 3) — same as RF.
    Batches tokenisation for GPU efficiency.
    """
    counts   = {'total': 0, 'parse_errors': 0,
                'no_qs': 0, 'scored': 0,
                'attack': 0, 'benign': 0}
    buf_rows = []
    buf_qvs  = []
    all_rows = []

    # Collect all entries first, then batch-score
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        for raw_line in f:
            parsed = parse_log_line(raw_line)
            if parsed is None:
                counts['parse_errors'] += 1
                continue
            qv = extract_query_values(parsed['url'])
            counts['total'] += 1
            if qv is None:
                counts['no_qs'] += 1
                all_rows.append({'label': label, 'url': parsed['url'],
                                 'qv': None, 'score': None, 'tier': 'NO_QS',
                                 'method': parsed['method'], 'status': parsed['status']})
            else:
                buf_rows.append({'label': label, 'url': parsed['url'],
                                 'qv': qv, 'score': None, 'tier': None,
                                 'method': parsed['method'], 'status': parsed['status']})
                buf_qvs.append(qv)

    # Batch-score all query-value entries
    print(f'  Scoring {len(buf_qvs):,} entries in batches of {MB_BATCH}...')
    mb_model.eval()
    all_probs = []
    n_batches = (len(buf_qvs) - 1) // MB_BATCH + 1
    for i in range(0, len(buf_qvs), MB_BATCH):
        batch  = buf_qvs[i:i+MB_BATCH]
        inputs = mb_tokenizer(batch, truncation=True, padding=True,
                              max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(mb_model(**inputs).logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        batch_num = i // MB_BATCH + 1
        if batch_num % 50 == 0 or batch_num == n_batches:
            print(f'    Batch {batch_num}/{n_batches} ({batch_num/n_batches*100:.0f}%)', flush=True)

    for row, prob in zip(buf_rows, all_probs):
        prob = round(float(prob), 6)
        tier = 'ATTACK' if prob >= threshold else 'BENIGN'
        counts['attack' if tier == 'ATTACK' else 'benign'] += 1
        counts['scored'] += 1
        row['score'] = prob
        row['tier']  = tier
        all_rows.append(row)

    # Write CSV
    with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile,
            fieldnames=['label', 'url', 'qv', 'score', 'tier', 'method', 'status'])
        writer.writeheader()
        writer.writerows(all_rows)

    return counts


print('Pipeline evaluators defined.')

## 6. Run RF Pipeline

In [ ]:
print('=' * 60)
print(f'RF PIPELINE — threshold={RF_THRESHOLD}')
print('=' * 60)

print('\nProcessing attack log...')
t0 = time.perf_counter()
rf_attack_counts = evaluate_rf(
    ATTACK_LOG, 'attack',
    'results/metrics/19_rf_attack_scored.csv',
    RF_THRESHOLD
)
print(f'  Done in {time.perf_counter()-t0:.1f}s')
print(f'  Total: {rf_attack_counts["total"]:,} | '
      f'NO_QS: {rf_attack_counts["no_qs"]:,} | '
      f'Scored: {rf_attack_counts["scored"]:,}')
print(f'  ATTACK: {rf_attack_counts["attack"]:,} | '
      f'BENIGN: {rf_attack_counts["benign"]:,}')

print('\nProcessing benign log...')
t1 = time.perf_counter()
rf_benign_counts = evaluate_rf(
    BENIGN_LOG, 'benign',
    'results/metrics/19_rf_benign_scored.csv',
    RF_THRESHOLD
)
print(f'  Done in {time.perf_counter()-t1:.1f}s')
print(f'  Total: {rf_benign_counts["total"]:,} | '
      f'NO_QS: {rf_benign_counts["no_qs"]:,} | '
      f'Scored: {rf_benign_counts["scored"]:,}')
print(f'  ATTACK (FP): {rf_benign_counts["attack"]:,} | '
      f'BENIGN: {rf_benign_counts["benign"]:,}')

rf_total_time = time.perf_counter() - t0
print(f'\nRF total time: {rf_total_time:.1f}s')

## 7. Run MobileBERT Zero-Shot Pipeline

In [ ]:
print('=' * 60)
print(f'MOBILEBERT ZERO-SHOT PIPELINE — threshold={MB_THRESHOLD}')
print('=' * 60)

print('\nProcessing attack log...')
t0 = time.perf_counter()
mb_attack_counts = evaluate_mobilebert(
    ATTACK_LOG, 'attack',
    'results/metrics/19_mb_attack_scored.csv',
    MB_THRESHOLD
)
print(f'  Done in {time.perf_counter()-t0:.1f}s')
print(f'  Total: {mb_attack_counts["total"]:,} | '
      f'NO_QS: {mb_attack_counts["no_qs"]:,} | '
      f'Scored: {mb_attack_counts["scored"]:,}')
print(f'  ATTACK: {mb_attack_counts["attack"]:,} | '
      f'BENIGN: {mb_attack_counts["benign"]:,}')

print('\nProcessing benign log...')
t1 = time.perf_counter()
mb_benign_counts = evaluate_mobilebert(
    BENIGN_LOG, 'benign',
    'results/metrics/19_mb_benign_scored.csv',
    MB_THRESHOLD
)
print(f'  Done in {time.perf_counter()-t1:.1f}s')
print(f'  Total: {mb_benign_counts["total"]:,} | '
      f'NO_QS: {mb_benign_counts["no_qs"]:,} | '
      f'Scored: {mb_benign_counts["scored"]:,}')
print(f'  ATTACK (FP): {mb_benign_counts["attack"]:,} | '
      f'BENIGN: {mb_benign_counts["benign"]:,}')

mb_total_time = time.perf_counter() - t0
print(f'\nMobileBERT total time: {mb_total_time:.1f}s')

# Free GPU memory
del mb_model
torch.cuda.empty_cache()
print('MobileBERT unloaded from GPU.')

## 8. Results — Recall & Precision

In [ ]:
def compute_metrics(attack_counts, benign_counts, threshold, model_name):
    a_total    = attack_counts['total']
    a_no_qs    = attack_counts['no_qs']
    a_scored   = attack_counts['scored']
    a_detected = attack_counts['attack']
    a_missed   = attack_counts['benign']

    b_total  = benign_counts['total']
    b_fp     = benign_counts['attack']

    recall_all   = a_detected / a_total  if a_total  > 0 else 0
    recall_qs    = a_detected / a_scored if a_scored > 0 else 0
    fp_per_10k   = (b_fp / b_total) * 10000 if b_total > 0 else 0
    flagged      = a_detected + b_fp
    precision    = a_detected / flagged if flagged > 0 else 0

    return {
        'model':         model_name,
        'threshold':     threshold,
        'a_total':       a_total,
        'a_no_qs':       a_no_qs,
        'a_scored':      a_scored,
        'a_detected':    a_detected,
        'a_missed':      a_missed,
        'recall_all':    recall_all,
        'recall_qs':     recall_qs,
        'b_total':       b_total,
        'b_fp':          b_fp,
        'fp_per_10k':    fp_per_10k,
        'precision':     precision,
    }

rf_metrics = compute_metrics(rf_attack_counts, rf_benign_counts, RF_THRESHOLD, 'RF')
mb_metrics = compute_metrics(mb_attack_counts, mb_benign_counts, MB_THRESHOLD, 'MobileBERT')

W = 72
print('=' * W)
print('PIPELINE EVALUATION — CSIC 2010 (Apache log format)')
print('RF (Way 3, threshold=0.82) vs MobileBERT zero-shot (Way 3, threshold=0.99)')
print('=' * W)
print(f'{"Metric":<40} {"RF":>14} {"MobileBERT":>14}')
print('-' * W)
print(f'{"Threshold":<40} {rf_metrics["threshold"]:>14.2f} {mb_metrics["threshold"]:>14.2f}')
print()
print(f'{"Attack entries (total)":<40} {rf_metrics["a_total"]:>14,} {mb_metrics["a_total"]:>14,}')
print(f'{"  Scored (has query string)":<40} {rf_metrics["a_scored"]:>14,} {mb_metrics["a_scored"]:>14,}')
print(f'{"  NO_QS (POST-body, not scoreable)":<40} {rf_metrics["a_no_qs"]:>14,} {mb_metrics["a_no_qs"]:>14,}')
print(f'{"  Detected":<40} {rf_metrics["a_detected"]:>14,} {mb_metrics["a_detected"]:>14,}')
print(f'{"  Missed":<40} {rf_metrics["a_missed"]:>14,} {mb_metrics["a_missed"]:>14,}')
print()
print(f'{"Recall — all entries":<40} {rf_metrics["recall_all"]:>14.4f} {mb_metrics["recall_all"]:>14.4f}')
print(f'{"Recall — scored (QS-only)":<40} {rf_metrics["recall_qs"]:>14.4f} {mb_metrics["recall_qs"]:>14.4f}')
print()
print(f'{"Benign entries (total)":<40} {rf_metrics["b_total"]:>14,} {mb_metrics["b_total"]:>14,}')
print(f'{"False positives":<40} {rf_metrics["b_fp"]:>14,} {mb_metrics["b_fp"]:>14,}')
print(f'{"FP/10k benign":<40} {rf_metrics["fp_per_10k"]:>14.4f} {mb_metrics["fp_per_10k"]:>14.4f}')
print()
print(f'{"Precision":<40} {rf_metrics["precision"]:>14.4f} {mb_metrics["precision"]:>14.4f}')
print('=' * W)

## 9. Recall by HTTP Method

In [ ]:
rf_attack_df = pd.read_csv('results/metrics/19_rf_attack_scored.csv')
mb_attack_df = pd.read_csv('results/metrics/19_mb_attack_scored.csv')

print('RECALL BY HTTP METHOD')
print()

for model_name, df in [('RF', rf_attack_df), ('MobileBERT', mb_attack_df)]:
    print(f'{model_name}:')
    print(f'  {"Method":<8} {"Total":>8} {"Scored":>8} {"Detected":>10} {"Recall":>8}')
    print('  ' + '-' * 48)
    for method in sorted(df['method'].unique()):
        sub      = df[df['method'] == method]
        scored   = sub[sub['tier'] != 'NO_QS']
        detected = sub[sub['tier'] == 'ATTACK']
        recall   = len(detected) / len(sub) if len(sub) > 0 else 0
        print(f'  {method:<8} {len(sub):>8,} {len(scored):>8,} {len(detected):>10,} {recall:>8.4f}')
    print()

## 10. Attack Type Breakdown

In [ ]:
rf_attack_df['attack_type'] = rf_attack_df['qv'].apply(categorise)
mb_attack_df['attack_type'] = mb_attack_df['qv'].apply(categorise)

print('ATTACK TYPE BREAKDOWN — DETECTED ENTRIES')
print()

all_types = sorted(set(
    rf_attack_df['attack_type'].unique().tolist() +
    mb_attack_df['attack_type'].unique().tolist()
))

print(f'{"Attack Type":<28} {"RF Total":>9} {"RF Det":>8} {"RF Rec":>8} '
      f'{"MB Total":>9} {"MB Det":>8} {"MB Rec":>8}')
print('-' * 82)

for atype in all_types:
    rf_sub = rf_attack_df[rf_attack_df['attack_type'] == atype]
    mb_sub = mb_attack_df[mb_attack_df['attack_type'] == atype]
    if len(rf_sub) == 0 and len(mb_sub) == 0:
        continue
    rf_det = len(rf_sub[rf_sub['tier'] == 'ATTACK'])
    mb_det = len(mb_sub[mb_sub['tier'] == 'ATTACK'])
    rf_rec = rf_det / len(rf_sub) if len(rf_sub) > 0 else 0
    mb_rec = mb_det / len(mb_sub) if len(mb_sub) > 0 else 0
    print(f'{atype:<28} {len(rf_sub):>9,} {rf_det:>8,} {rf_rec:>8.4f} '
          f'{len(mb_sub):>9,} {mb_det:>8,} {mb_rec:>8.4f}')

## 11. SQLi-Specific Recall

In [ ]:
rf_sqli_df = rf_attack_df[rf_attack_df['attack_type'].isin(SQLI_TYPES)]
mb_sqli_df = mb_attack_df[mb_attack_df['attack_type'].isin(SQLI_TYPES)]

rf_sqli_det    = rf_sqli_df[rf_sqli_df['tier'] == 'ATTACK']
mb_sqli_det    = mb_sqli_df[mb_sqli_df['tier'] == 'ATTACK']

rf_sqli_recall = len(rf_sqli_det) / len(rf_sqli_df) if len(rf_sqli_df) > 0 else 0
mb_sqli_recall = len(mb_sqli_det) / len(mb_sqli_df) if len(mb_sqli_df) > 0 else 0

print('SQLi-SPECIFIC RECALL (pipeline-level)')
print()
print(f'{"Metric":<40} {"RF":>12} {"MobileBERT":>12}')
print('-' * 66)
print(f'{"SQLi entries (pipeline-scored)":<40} {len(rf_sqli_df):>12,} {len(mb_sqli_df):>12,}')
print(f'{"SQLi detected":<40} {len(rf_sqli_det):>12,} {len(mb_sqli_det):>12,}')
print(f'{"SQLi recall":<40} {rf_sqli_recall:>12.4f} {mb_sqli_recall:>12.4f}')
print()

print(f'{"SQLi Type":<28} {"RF Total":>9} {"RF Det":>8} {"RF Rec":>8} '
      f'{"MB Det":>8} {"MB Rec":>8}')
print('-' * 75)
for t in SQLI_TYPES:
    rf_s = rf_attack_df[rf_attack_df['attack_type'] == t]
    mb_s = mb_attack_df[mb_attack_df['attack_type'] == t]
    if len(rf_s) == 0:
        continue
    rf_d = len(rf_s[rf_s['tier'] == 'ATTACK'])
    mb_d = len(mb_s[mb_s['tier'] == 'ATTACK'])
    rf_r = rf_d / len(rf_s)
    mb_r = mb_d / len(mb_s) if len(mb_s) > 0 else 0
    print(f'{t:<28} {len(rf_s):>9,} {rf_d:>8,} {rf_r:>8.4f} {mb_d:>8,} {mb_r:>8.4f}')

## 12. False Positives on Benign Traffic

In [ ]:
rf_benign_df = pd.read_csv('results/metrics/19_rf_benign_scored.csv')
mb_benign_df = pd.read_csv('results/metrics/19_mb_benign_scored.csv')

rf_fps = rf_benign_df[rf_benign_df['tier'] == 'ATTACK'].sort_values('score', ascending=False)
mb_fps = mb_benign_df[mb_benign_df['tier'] == 'ATTACK'].sort_values('score', ascending=False)

print('FALSE POSITIVES ON BENIGN TRAFFIC')
print()
print(f'{"Model":<14} {"FP count":>10} {"FP/10k":>10}')
print('-' * 40)
print(f'{"RF":<14} {len(rf_fps):>10,} {(len(rf_fps)/len(rf_benign_df))*10000:>10.4f}')
print(f'{"MobileBERT":<14} {len(mb_fps):>10,} {(len(mb_fps)/len(mb_benign_df))*10000:>10.4f}')
print()

for model_name, fps in [('RF', rf_fps), ('MobileBERT', mb_fps)]:
    print(f'{model_name} false positives (top 20):')
    if len(fps) == 0:
        print('  None.')
    else:
        print(f'  {"Score":>8} {"Method":<6} Query Values')
        print('  ' + '-' * 70)
        for _, row in fps.head(20).iterrows():
            qv = str(row['qv'])[:55] if row['qv'] else '-'
            print(f'  {row["score"]:>8.4f} {row["method"]:<6} {qv}')
        if len(fps) > 20:
            print(f'  ... and {len(fps)-20} more')
    print()

rf_fps.to_csv('results/metrics/19_rf_false_positives.csv', index=False)
mb_fps.to_csv('results/metrics/19_mb_false_positives.csv', index=False)
print('Saved: 19_rf_false_positives.csv, 19_mb_false_positives.csv')

## 13. Final Comparison Summary

In [ ]:
W = 72
print('=' * W)
print('NOTEBOOK 19 — COMPLETE')
print('Pipeline-Level Evaluation: RF vs MobileBERT Zero-Shot')
print('CSIC 2010 — Apache log format — Way 3 (query values)')
print('=' * W)
print()
print(f'{"Metric":<40} {"RF":>14} {"MobileBERT":>14}')
print('-' * W)
print(f'{"Threshold":<40} {RF_THRESHOLD:>14.2f} {MB_THRESHOLD:>14.2f}')
print(f'{"Strategy":<40} {"Way 3":>14} {"Way 3":>14}')
print(f'{"Infrastructure":<40} {"CPU":>14} {"GPU":>14}')
print()
print(f'{"Recall — all entries":<40} {rf_metrics["recall_all"]:>14.4f} {mb_metrics["recall_all"]:>14.4f}')
print(f'{"Recall — scored (QS only)":<40} {rf_metrics["recall_qs"]:>14.4f} {mb_metrics["recall_qs"]:>14.4f}')
print(f'{"SQLi-specific recall (pipeline)":<40} {rf_sqli_recall:>14.4f} {mb_sqli_recall:>14.4f}')
print(f'{"FP/10k benign":<40} {rf_metrics["fp_per_10k"]:>14.4f} {mb_metrics["fp_per_10k"]:>14.4f}')
print(f'{"Precision":<40} {rf_metrics["precision"]:>14.4f} {mb_metrics["precision"]:>14.4f}')
print(f'{"Pipeline time (sec)":<40} {rf_total_time:>14.1f} {mb_total_time:>14.1f}')
print()
print(f'{"NO_QS (POST-body, out of scope)":<40} {rf_metrics["a_no_qs"]:>14,} {mb_metrics["a_no_qs"]:>14,}')
print('=' * W)

# Save summary
summary = {
    'dataset':         'CSIC 2010 (Apache log format)',
    'pipeline':        'parse_log_line → extract_query_values → score',
    'strategy':        'Way 3 — query parameter values joined',
    'rf': {
        'threshold':       RF_THRESHOLD,
        'attack_total':    rf_metrics['a_total'],
        'attack_scored':   rf_metrics['a_scored'],
        'attack_no_qs':    rf_metrics['a_no_qs'],
        'detected':        rf_metrics['a_detected'],
        'recall_all':      round(rf_metrics['recall_all'],   4),
        'recall_qs':       round(rf_metrics['recall_qs'],    4),
        'sqli_recall':     round(rf_sqli_recall,             4),
        'fp_count':        rf_metrics['b_fp'],
        'fp_per_10k':      round(rf_metrics['fp_per_10k'],   4),
        'precision':       round(rf_metrics['precision'],    4),
        'pipeline_sec':    round(rf_total_time,              1),
    },
    'mobilebert': {
        'model':           'cssupport/mobilebert-sql-injection-detect',
        'mode':            'zero-shot',
        'threshold':       MB_THRESHOLD,
        'attack_total':    mb_metrics['a_total'],
        'attack_scored':   mb_metrics['a_scored'],
        'attack_no_qs':    mb_metrics['a_no_qs'],
        'detected':        mb_metrics['a_detected'],
        'recall_all':      round(mb_metrics['recall_all'],   4),
        'recall_qs':       round(mb_metrics['recall_qs'],    4),
        'sqli_recall':     round(mb_sqli_recall,             4),
        'fp_count':        mb_metrics['b_fp'],
        'fp_per_10k':      round(mb_metrics['fp_per_10k'],   4),
        'precision':       round(mb_metrics['precision'],    4),
        'pipeline_sec':    round(mb_total_time,              1),
    }
}

with open('results/metrics/19_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nSaved: results/metrics/19_summary.json')
print(json.dumps(summary, indent=2))